# Tutorial: Estimating Social Contact Matrices with Prem Model

In [ ]:
from jax.random import PRNGKey

from cntmosaic.utils import AgeBins
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import ParticipantGenerator, MatrixGenerator, ContactGenerator, Subgroup
from cntmosaic.vis import plot_mosaic, plot_mosaic_pixilated

from cntmosaic.models import Prem
from cntmosaic.analysis import ModelSummariserPrem, ModelEvaluatorPrem

import altair as alt

## 1. Introduction
The `Prem` class in `cntmosaic` is a Python implementation of the model proposed by Prem et al. (2017) for estimating social contact matrices from survey data. This tutorial will guide you through the process of using `Prem` to analyze contact survey data.

## 2. Synthetic Data Generation
We'll use `cntmosaic`'s data generation tools to create realistic synthetic contact data. For details, refer to the "Tutorial: Generating Synthetic Contact Data".

We first load the contact pattern templates and population data:

In [ ]:
templates = load_template_patterns(country='United_States')
df_age_dist = load_age_distribution(country='United_States')

age_dist = df_age_dist.P.values

We define a subgroup with 1000 participants with the `Subgroup` class:

In [ ]:
general = Subgroup(
  n=1000,                       # Generate 1000 participants
  age_dist=age_dist,            # Use this age distribution
  mean_cint_margin=15.0,        # Average 15 contacts per person
  label='general'
)

We randomly simulate a contact intensity matrix using the `MatrixGenerator` class.

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
contact_matrix = matrix_gen.generate_single(
  subgroup=general,
  seed=42   # For reproducibility
)

print("Matrix shape:", contact_matrix.shape)
print("Average contacts per person:", contact_matrix.sum(axis=0).mean())

Visualize the generated contact matrix with `plot_mosaic` function:

In [ ]:
plot_mosaic(contact_matrix, title='Synthetic Contact Intensity Matrix')

The `ParticipantGenerator` class is then used to simulate participant data based on the defined subgroup.

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(general)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

The `ContactGenerator` class simulates contact data for the participants:

In [ ]:
cnt_gen = ContactGenerator(
  df_part=df_part,
  cint_matrices=contact_matrix,
  model='poisson'
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

Plot the empirical contact matrix from the simulated contact data:

In [ ]:
emp_cint_matrix = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)
plot_mosaic(emp_cint_matrix, title='Empirical Contact Matrix', zlabel='Intensity')

## 3. Estimating Contact Matrices with the Prem Model

### 3.1 Data Preparation

Besides the participant, contact, and population age distribution dataframes, the `SocialMix` class requires information about the age bins used for grouping the participants and contacts. We give these information in the form of an `AgeBins` object. Here, we define 5-year age bins from 0 to 80 years. The participant dataframe must also contain a column named `age_part` (or `age_grp_part` if age groups are pre-defined).

In [ ]:
age_bins = AgeBins(0, 80, 5)
df_part['age_part'] = df_part['age_group']

### 3.2 Basic Matrix Estimation

In [ ]:
# Initialize Prem Model
prem = Prem(
  df_part=df_part,
  df_cnt=df_cnt,
  age_bins=age_bins
)

#### Fast Inference using SVI
For quick estimation of contact matrices, we can use Stochastic Variational Inference (SVI) via the `run_inference_svi` method.

In [ ]:
from numpyro.infer.autoguide import AutoNormal

rng_key = PRNGKey(0)
guide = AutoNormal(prem.model)

prem.run_inference_svi(
  rng_key,
  guide=guide,
  num_steps=5000,
  peak_lr=0.01
)

In [ ]:
# Initialize summariser
summariser = ModelSummariserPrem(prem, df_age_dist)

In [ ]:
# Summarise the posterior contact intensity matrix
summary = summariser.summarise_cint()

# The output is a dictionary with keys 'median', 'lower', 'upper', and alpha
cint_median = summary['median']
cint_lower = summary['lower']
cint_upper = summary['upper']

# Plot the inferred contact intensity matrices
plt_median = plot_mosaic_pixilated(
  cint_median, age_bins, title='Inferred Contact Intensity Matrix (Median)', zlabel='Intensity',
  ylabel=None
)
plt_lower = plot_mosaic_pixilated(
  cint_lower, age_bins, title='Inferred Contact Intensity Matrix (2.5th Percentile)', zlabel='Intensity'
)
plt_upper = plot_mosaic_pixilated(
  cint_upper, age_bins, title='Inferred Contact Intensity Matrix (97.5th Percentile)', zlabel='Intensity',
  ylabel=None
)

alt.hconcat(plt_lower, plt_median, plt_upper)

In the Prem model, the reciprocity contraint is applied to the posterior samples of the contact intensity matrices.

In [ ]:
summary_sym = summariser.summarise_cint(return_symmetrized=True)
cint_median_sym = summary_sym['median']

plt_sym = plot_mosaic_pixilated(
  cint_median_sym, age_bins, title='Symmetrized Contact Intensity Matrix (Median)', zlabel='Intensity',
  ylabel=None
)

alt.hconcat(plt_median, plt_sym)

We can evaluate the estimated contact matrix against the true contact matrix using the `ModelEvaluatorPrem` class. Here, we include extended metrics in the evaluation.

In [ ]:
evaluator = ModelEvaluatorPrem(summariser, contact_matrix)

In [ ]:
evaluator.evaluate(include_extended=True)

We can also compare the metric where the symmetrized contact intensity matrix is used.

In [ ]:
evaluator = ModelEvaluatorPrem(
  summariser,
  contact_matrix,
  symmetric=True
)

In [ ]:
evaluator.evaluate(include_extended=True, symmetric=True)